
# 2D Euler–Bernoulli Frame Element

This notebook explains the frame analysis using the **2D Euler–Bernoulli frame element**.

All calculations are performed in **SI units**.

Thus the values in figure in SI units as below:


$$
E=10^6\;\mathrm{psi}=6.9\times10^9\;\mathrm{Pa}
$$

$$
L = 1\; \mathrm{in} = 0.025~\mathrm{m}
$$
$$
A=1\;\mathrm{in^2}=6.45\times10^{-4}\;\mathrm{m^2}
$$

$$
I=1\;\mathrm{in^4}=4.16\times10^{-7}\;\mathrm{m^4}
$$

The geometry becomes

$$
144\;\mathrm{in}=3.6576\;\mathrm{m},\qquad
108\;\mathrm{in}=2.7432\;\mathrm{m},\qquad
72\;\mathrm{in}=1.8288\;\mathrm{m}.
$$

The load parameter is taken as

$$
P=1\;\mathrm{N}
$$


## 1. Nodes, elements, and degrees of freedom

We use three nodes:

$ \text{Node 1}: (0,0) ; \text{Node 2}: (0,3.6576) ; \text{Node 3}: (3.6576,6.4008) $

The element continuity is given as
$$
\text{Element 1}: 1\rightarrow2,
\qquad
\text{Element 2}: 2\rightarrow3.
$$

Each node has three degrees of freedom:

$$
\mathbf q_i=
\begin{bmatrix}
u_i\\
w_i\\
\theta_i
\end{bmatrix}.
$$

The complete displacement vector is

$$
\mathbf Q=
\begin{bmatrix}
u_1&w_1&\theta_1&
u_2&w_2&\theta_2&
u_3&w_3&\theta_3
\end{bmatrix}^T.
$$

Thus, the structure has nine global DOFs.

Nodes 1 and 3 are fixed:

$$
u_1=w_1=\theta_1=0,
\qquad
u_3=w_3=\theta_3=0.
$$


In [ ]:

import numpy as np

np.set_printoptions(precision=5, suppress=True)

E = 1.0e6 * 6894.757293168       # Pa
A = 10.0 * (0.0254**2)           # m^2
I = 10.0 * (0.0254**4)           # m^4

P = 1.0                           # N

coords = np.array([
    [0.0,       0.0],
    [0.0,       3.6576],
    [3.6576,    6.4008]
])

elements = [(0,1), (1,2)]

print(f"E = {E:.6e} Pa")
print(f"A = {A:.6e} m^2")
print(f"I = {I:.6e} m^4")


E = 6.894757e+09 Pa
A = 6.451600e-03 m^2
I = 4.162314e-06 m^4


### Let's examine one frame element which is combination of bar behavior and beam behavior

A 2D frame element carries both axial and bending actions.

### Bar contribution

$$
\bar{\mathbf K}_{bar}
=
\frac{EA}{L}
\begin{bmatrix}
1&-1\\
-1&1
\end{bmatrix}.
$$

This acts on the local axial DOFs

$$
\bar u_1,\qquad \bar u_2.
$$

### EBBT contribution from the Lecture \#12 and \#13

$$
\bar{\mathbf K}_{beam}
=
\frac{EI}{L^3}
\begin{bmatrix}
12&6L&-12&6L\\
6L&4L^2&-6L&2L^2\\
-12&-6L&12&-6L\\
6L&2L^2&-6L&4L^2
\end{bmatrix}.
$$

This acts on

$$
\bar w_1,\qquad
\bar\theta_1,\qquad
\bar w_2,\qquad
\bar\theta_2.
$$

Using the local DOF order

$$
\bar{\mathbf q}^{\,e}
=
\begin{bmatrix}
\bar u_1&
\bar w_1&
\bar\theta_1&
\bar u_2&
\bar w_2&
\bar\theta_2
\end{bmatrix}^T,
$$

the combined frame matrix is

$$
\bar{\mathbf K}^{\,e}
=
\begin{bmatrix}
EA/L&0&0&-EA/L&0&0\\
0&12EI/L^3&6EI/L^2&0&-12EI/L^3&6EI/L^2\\
0&6EI/L^2&4EI/L&0&-6EI/L^2&2EI/L\\
-EA/L&0&0&EA/L&0&0\\
0&-12EI/L^3&-6EI/L^2&0&12EI/L^3&-6EI/L^2\\
0&6EI/L^2&2EI/L&0&-6EI/L^2&4EI/L
\end{bmatrix}.
$$


In [ ]:

def frame_local_stiffness(E, A, I, L):
    k = np.array([
        [ E*A/L,             0.0,              0.0, -E*A/L,             0.0,              0.0],
        [ 0.0,       12*E*I/L**3,       6*E*I/L**2,    0.0,    -12*E*I/L**3,       6*E*I/L**2],
        [ 0.0,        6*E*I/L**2,          4*E*I/L,    0.0,     -6*E*I/L**2,          2*E*I/L],
        [-E*A/L,             0.0,              0.0,  E*A/L,             0.0,              0.0],
        [ 0.0,      -12*E*I/L**3,      -6*E*I/L**2,    0.0,     12*E*I/L**3,      -6*E*I/L**2],
        [ 0.0,        6*E*I/L**2,          2*E*I/L,    0.0,     -6*E*I/L**2,          4*E*I/L]
    ], dtype=float)
    return k



## Let's apply Coordinate transformation

For an element inclined at angle $\alpha$ from the global $x$ axis,

$$
c=\cos\alpha,
\qquad
s=\sin\alpha.
$$

For one node,

$$
\begin{bmatrix}
\bar u\\
\bar w\\
\bar\theta
\end{bmatrix}
=
\begin{bmatrix}
c&s&0\\
-s&c&0\\
0&0&1
\end{bmatrix}
\begin{bmatrix}
u\\
w\\
\theta
\end{bmatrix}.
$$

For the complete element,

$$
\bar{\mathbf q}^{\,e}
=
\mathbf T^e\mathbf q^e.
$$

The global element stiffness matrix is

$$
\mathbf K^e
=
(\mathbf T^e)^T
\bar{\mathbf K}^{\,e}
\mathbf T^e.
$$


In [ ]:

def element_geometry(node_i, node_j):
    x1, z1 = coords[node_i]
    x2, z2 = coords[node_j]
    dx = x2 - x1
    dz = z2 - z1
    L = np.sqrt(dx**2 + dz**2)
    c = dx/L
    s = dz/L
    alpha = np.arctan2(dz, dx)
    return L, c, s, alpha

def transformation_matrix(c, s):
    return np.array([
        [ c,  s, 0.0, 0.0, 0.0, 0.0],
        [-s,  c, 0.0, 0.0, 0.0, 0.0],
        [0.0,0.0, 1.0, 0.0, 0.0, 0.0],
        [0.0,0.0, 0.0,  c,   s, 0.0],
        [0.0,0.0, 0.0, -s,   c, 0.0],
        [0.0,0.0, 0.0,0.0, 0.0, 1.0]
    ])

def get_element_data(node_i, node_j):
    L, c, s, alpha = element_geometry(node_i, node_j)
    k_local = frame_local_stiffness(E, A, I, L)
    T = transformation_matrix(c, s)
    k_global = T.T @ k_local @ T
    return L, c, s, alpha, k_local, T, k_global



## Analyzing Element 2 as a single-element example

For Element 2,

$$
\Delta x=3.6576\;\mathrm{m},
\qquad
\Delta z=2.7432\;\mathrm{m}.
$$

Therefore,

$$
L_2=4.572\;\mathrm{m}
$$

and

$$
c=0.8,
\qquad
s=0.6,
\qquad
\alpha_2=36.87^\circ.
$$

The local matrix changes after coordinate transformation as below:


In [ ]:

L2, c2, s2, alpha2, k2_local, T2, k2_global = get_element_data(1,2)

print(f"L2 = {L2:.6f} m")
print(f"alpha2 = {np.degrees(alpha2):.6f} degrees")
print(f"c = {c2:.6f}, s = {s2:.6f}")

print("\nLocal stiffness matrix, Element 2")
print(k2_local)

print("\nTransformation matrix, Element 2")
print(T2)

print("\nGlobal stiffness matrix, Element 2")
print(k2_global)


L2 = 4.572000 m
alpha2 = 36.869898 degrees
c = 0.800000, s = 0.600000

Local stiffness matrix, Element 2
[[ 9729268.6248         0.             0.      -9729268.6248
         0.             0.     ]
 [       0.          3603.43282     8237.44744        0.
     -3603.43282     8237.44744]
 [       0.          8237.44744    25107.73978        0.
     -8237.44744    12553.86989]
 [-9729268.6248         0.             0.       9729268.6248
         0.             0.     ]
 [       0.         -3603.43282    -8237.44744        0.
      3603.43282    -8237.44744]
 [       0.          8237.44744    12553.86989        0.
     -8237.44744    25107.73978]]

Transformation matrix, Element 2
[[ 0.8  0.6  0.   0.   0.   0. ]
 [-0.6  0.8  0.   0.   0.   0. ]
 [ 0.   0.   1.   0.   0.   0. ]
 [ 0.   0.   0.   0.8  0.6  0. ]
 [ 0.   0.   0.  -0.6  0.8  0. ]
 [ 0.   0.   0.   0.   0.   1. ]]

Global stiffness matrix, Element 2
[[ 6228029.15569  4668319.29215    -4942.46846 -6228029.15569
  -4668319.2921

The local matrix clearly separates axial and bending terms.

After transformation, the inclined member develops coupling between global $x$ and $z$ translations because a global translation generally has both a component parallel to the member and a component perpendicular to it.



## Load vectors

### Distributed load on Element 1

The original distributed load is

$$
q=\frac{P}{72\;\mathrm{in}}.
$$

Since

$$
72\;\mathrm{in}=1.83\;\mathrm{m},
$$

the SI form is

$$
q=\frac{P}{1.83}\;\mathrm{N/m}.
$$

For $P=1$ N,

$$
q=0.55\;\mathrm{N/m}.
$$

For a constant local transverse load $\bar q$,

$$
\bar{\mathbf f}_{q}^{\,e}
=
\begin{bmatrix}
0\\
\bar qL/2\\
\bar qL^2/12\\
0\\
\bar qL/2\\
-\bar qL^2/12
\end{bmatrix}.
$$

### Point load on Element 2

The $4P$ downward load is located at the midpoint of Element 2.

Global force vector is transformed to the local element system. The local axial component uses **linear bar interpolation**, while the local transverse component uses the **cubic Hermite beam interpolation** functions.


In [ ]:

def uniform_load_global(node_i, node_j, qx=0.0, qz=0.0):
    L, c, s, alpha, k_local, T, k_global = get_element_data(node_i, node_j)

    R = np.array([[c, s], [-s, c]])
    qu, qw = R @ np.array([qx, qz])

    f_local = np.zeros(6)

    # axial part
    f_local[0] += qu*L/2
    f_local[3] += qu*L/2

    # transverse Euler-Bernoulli part
    f_local[1] += qw*L/2
    f_local[2] += qw*L**2/12
    f_local[4] += qw*L/2
    f_local[5] += -qw*L**2/12

    f_global = T.T @ f_local
    return f_local, f_global

def point_load_global(node_i, node_j, a, Px=0.0, Pz=0.0):
    L, c, s, alpha, k_local, T, k_global = get_element_data(node_i, node_j)

    xi = a/L

    R = np.array([[c, s], [-s, c]])
    Pu, Pw = R @ np.array([Px, Pz])

    # bar shape functions
    Nu1 = 1.0 - xi
    Nu2 = xi

    # Hermite beam shape functions
    N1 = 1 - 3*xi**2 + 2*xi**3
    N2 = L*(xi - 2*xi**2 + xi**3)
    N3 = 3*xi**2 - 2*xi**3
    N4 = L*(-xi**2 + xi**3)

    f_local = np.array([
        Nu1*Pu,
        N1*Pw,
        N2*Pw,
        Nu2*Pu,
        N3*Pw,
        N4*Pw
    ])

    f_global = T.T @ f_local
    return f_local, f_global

q = P / 1.8288

f1_local, f1_global = uniform_load_global(
    0, 1, qx=q, qz=0.0
)

f2_local, f2_global = point_load_global(
    1, 2, a=L2/2, Px=0.0, Pz=-4.0*P
)

print("Element 1 local consistent load:")
print(f1_local)

print("\nElement 1 global consistent load:")
print(f1_global)

print("\nElement 2 local consistent load:")
print(f2_local)

print("\nElement 2 global consistent load:")
print(f2_global)


Element 1 local consistent load:
[ 0.     -1.     -0.6096  0.     -1.      0.6096]

Element 1 global consistent load:
[ 1.      0.     -0.6096  1.      0.      0.6096]

Element 2 local consistent load:
[-1.2    -1.6    -1.8288 -1.2    -1.6     1.8288]

Element 2 global consistent load:
[ 0.     -2.     -1.8288  0.     -2.      1.8288]



## Assemble the global stiffness matrix and global load vector

The global DOF numbering used is

$$
\begin{array}{c|ccc}
\text{Node}&u&w&\theta\\
\hline
1&0&1&2\\
2&3&4&5\\
3&6&7&8
\end{array}.
$$

Therefore,

$$
\text{Element 1 DOFs}=[0,1,2,3,4,5]
$$

and

$$
\text{Element 2 DOFs}=[3,4,5,6,7,8].
$$

Each element stiffness contribution is added into the corresponding rows and columns of the full global matrix.


In [ ]:
ndof = 9
K = np.zeros((ndof, ndof))
F = np.zeros(ndof)

def element_dofs(node_i, node_j):
    return [
        3*node_i, 3*node_i+1, 3*node_i+2,
        3*node_j, 3*node_j+1, 3*node_j+2
    ]

element_store = []

# Element 1
L1, c1, s1, alpha1, k1_local, T1, k1_global = get_element_data(0,1)
dofs1 = element_dofs(0,1)

for a in range(6):
    F[dofs1[a]] += f1_global[a]
    for b in range(6):
        K[dofs1[a], dofs1[b]] += k1_global[a,b]

element_store.append({
    "dofs": dofs1,
    "T": T1,
    "k_local": k1_local,
    "f_local": f1_local
})

# Element 2
dofs2 = element_dofs(1,2)

for a in range(6):
    F[dofs2[a]] += f2_global[a]
    for b in range(6):
        K[dofs2[a], dofs2[b]] += k2_global[a,b]

element_store.append({
    "dofs": dofs2,
    "T": T2,
    "k_local": k2_local,
    "f_local": f2_local
})

labels = [
    "u1","w1","theta1",
    "u2","w2","theta2",
    "u3","w3","theta3"
]

print("Global stiffness matrix K:")
print(K)

print("\nGlobal load vector F:")
for label, value in zip(labels, F):
    print(f"{label:>7s} : {value: .6e}")


Global stiffness matrix K:
[[     7037.95473         0.         -12871.01162     -7037.95473
          0.         -12871.01162         0.              0.
          0.     ]
 [        0.       12161585.781           0.              0.
  -12161585.781           0.              0.              0.
          0.     ]
 [   -12871.01162         0.          31384.67473     12871.01162
          0.          15692.33736         0.              0.
          0.     ]
 [    -7037.95473         0.          12871.01162   6235067.11043
    4668319.29215      7928.54316  -6228029.15569  -4668319.29215
      -4942.46846]
 [        0.      -12161585.781           0.        4668319.29215
   15666428.68294      6589.95795  -4668319.29215  -3504842.90194
       6589.95795]
 [   -12871.01162         0.          15692.33736      7928.54316
       6589.95795     56492.41451      4942.46846     -6589.95795
      12553.86989]
 [        0.              0.              0.       -6228029.15569
   -4668319.29215    


## Apply the boundary conditions to the full matrix

We have $ 9\times 9 $ system.

For every prescribed zero displacement DOF, we follow the following:

1. zero its row and column,
2. place $1$ on the diagonal,
3. set the corresponding load-vector entry to zero.

The fixed DOFs are

$$
[0,1,2,6,7,8].
$$

The "hacked" system is solved below.

In [ ]:
K_original = K.copy()
F_original = F.copy()

fixed_dofs = [0,1,2,6,7,8]

K_bc = K.copy()
F_bc = F.copy()

for dof in fixed_dofs:
    K_bc[dof,:] = 0.0
    K_bc[:,dof] = 0.0
    K_bc[dof,dof] = 1.0
    F_bc[dof] = 0.0

print("Full 9x9 matrix after applying boundary conditions:")
print(K_bc)

print("\nFull modified load vector:")
for label, value in zip(labels, F_bc):
    print(f"{label:>7s} : {value: .6e}")


Full 9x9 matrix after applying boundary conditions:
[[       1.             0.             0.             0.
         0.             0.             0.             0.
         0.     ]
 [       0.             1.             0.             0.
         0.             0.             0.             0.
         0.     ]
 [       0.             0.             1.             0.
         0.             0.             0.             0.
         0.     ]
 [       0.             0.             0.       6235067.11043
   4668319.29215     7928.54316        0.             0.
         0.     ]
 [       0.             0.             0.       4668319.29215
  15666428.68294     6589.95795        0.             0.
         0.     ]
 [       0.             0.             0.          7928.54316
      6589.95795    56492.41451        0.             0.
         0.     ]
 [       0.             0.             0.             0.
         0.             0.             1.             0.
         0.     ]
 [       


## Solve the system

We solve

$$
\mathbf K_{BC}\mathbf Q
=
\mathbf F_{BC}.
$$



In [ ]:
Q = np.linalg.solve(K_bc, F_bc)

print("Displacement vector Q:")
for label, value in zip(labels, Q):
    unit = "rad" if "theta" in label else "m"
    print(f"{label:>7s} : {value: .8e} {unit}")

print("\nNode 2 results:")
print(f"u2     = {Q[3]: .8e} m")
print(f"w2     = {Q[4]: .8e} m")
print(f"theta2 = {Q[5]: .8e} rad")


Displacement vector Q:
     u1 :  0.00000000e+00 m
     w1 :  0.00000000e+00 m
 theta1 :  0.00000000e+00 rad
     u2 :  3.56077891e-07 m
     w2 : -2.24678293e-07 m
 theta2 : -2.16054238e-05 rad
     u3 :  0.00000000e+00 m
     w3 :  0.00000000e+00 m
 theta3 :  0.00000000e+00 rad

Node 2 results:
u2     =  3.56077891e-07 m
w2     = -2.24678293e-07 m
theta2 = -2.16054238e-05 rad



## 9. Support reactions

Use the original, unmodified global matrix:

$$
\mathbf R
=
\mathbf K\mathbf Q-\mathbf F.
$$


In [ ]:
R = K_original @ Q - F_original

print("Reaction vector:")
for label, value in zip(labels, R):
    unit = "N*m" if "theta" in label else "N"
    print(f"{label:>7s} : {value: .8e} {unit}")

print("\nSupport reactions only:")
for dof in fixed_dofs:
    unit = "N*m" if "theta" in labels[dof] else "N"
    print(f"{labels[dof]:>7s}: {R[dof]: .8e} {unit}")


Reaction vector:
     u1 : -7.24422399e-01 N
     w1 :  2.73244433e+00 N
 theta1 :  2.75143483e-01 N*m
     u2 :  0.00000000e+00 N
     w2 :  4.44089210e-16 N
 theta2 :  0.00000000e+00 N*m
     u3 : -1.27557760e+00 N
     w3 :  1.26755567e+00 N
 theta3 : -2.10327220e+00 N*m

Support reactions only:
     u1: -7.24422399e-01 N
     w1:  2.73244433e+00 N
 theta1:  2.75143483e-01 N*m
     u3: -1.27557760e+00 N
     w3:  1.26755567e+00 N
 theta3: -2.10327220e+00 N*m



## 10. Recover local element forces

For each element,

$$
\bar{\mathbf q}^{\,e}
=
\mathbf T^e\mathbf q^e
$$

and

$$
\bar{\mathbf r}^{\,e}
=
\bar{\mathbf k}^{\,e}\bar{\mathbf q}^{\,e}
-
\bar{\mathbf f}^{\,e}.
$$

The local end-force ordering is

$$
\bar{\mathbf r}^{\,e}
=
\begin{bmatrix}
N_1&
V_1&
M_1&
N_2&
V_2&
M_2
\end{bmatrix}^T.
$$


In [ ]:
for e, data in enumerate(element_store, start=1):
    dofs = data["dofs"]
    q_global_e = Q[dofs]
    q_local_e = data["T"] @ q_global_e
    r_local_e = data["k_local"] @ q_local_e - data["f_local"]

    print(f"\nElement {e}")
    print("-"*50)
    print("Local displacement vector:")
    print(q_local_e)

    print("\nLocal end-force vector [N1, V1, M1, N2, V2, M2]:")
    print(r_local_e)

    print("\nIndividual element forces:")
    force_labels = ["N1", "V1", "M1", "N2", "V2", "M2"]
    force_units  = ["N", "N", "N*m", "N", "N", "N*m"]
    for name, value, unit in zip(force_labels, r_local_e, force_units):
        print(f"{name:>3s} = {value: .8e} {unit}")



Element 1
--------------------------------------------------
Local displacement vector:
[ 0.       0.       0.      -0.      -0.      -0.00002]

Local end-force vector [N1, V1, M1, N2, V2, M2]:
[ 2.73244  0.72442  0.27514 -2.73244  1.27558 -1.2831 ]

Individual element forces:
 N1 =  2.73244433e+00 N
 V1 =  7.24422399e-01 N
 M1 =  2.75143483e-01 N*m
 N2 = -2.73244433e+00 N
 V2 =  1.27557760e+00 N
 M2 = -1.28309612e+00 N*m

Element 2
--------------------------------------------------
Local displacement vector:
[ 0.      -0.      -0.00002  0.       0.       0.     ]

Local end-force vector [N1, V1, M1, N2, V2, M2]:
[ 2.65993  1.42061  1.2831  -0.25993  1.77939 -2.10327]

Individual element forces:
 N1 =  2.65992868e+00 N
 V1 =  1.42060890e+00 N
 M1 =  1.28309612e+00 N*m
 N2 = -2.59928680e-01 N
 V2 =  1.77939110e+00 N
 M2 = -2.10327220e+00 N*m



## 11. Global force-equilibrium check

The sum of applied forces and support reactions should be approximately zero in both global directions.


In [ ]:

Fx_applied = F_original[[0,3,6]].sum()
Fz_applied = F_original[[1,4,7]].sum()

Fx_reaction = R[[0,3,6]].sum()
Fz_reaction = R[[1,4,7]].sum()

print(f"Applied Fx   = {Fx_applied: .8e} N")
print(f"Reaction Fx  = {Fx_reaction: .8e} N")
print(f"Fx residual  = {Fx_applied + Fx_reaction: .8e} N")

print()

print(f"Applied Fz   = {Fz_applied: .8e} N")
print(f"Reaction Fz  = {Fz_reaction: .8e} N")
print(f"Fz residual  = {Fz_applied + Fz_reaction: .8e} N")


Applied Fx   =  2.00000000e+00 N
Reaction Fx  = -2.00000000e+00 N
Fx residual  = -4.44089210e-16 N

Applied Fz   = -4.00000000e+00 N
Reaction Fz  =  4.00000000e+00 N
Fz residual  =  0.00000000e+00 N



## Summary

What we did is

$$
\text{local bar + beam stiffness}
\rightarrow
\text{coordinate transformation}
\rightarrow
\text{global element stiffness}
\rightarrow
\text{assembly}
\rightarrow
\text{full global system}
\rightarrow
\text{boundary conditions}
\rightarrow
\text{solution}
\rightarrow
\text{reactions and element forces}.
$$
